# RGB Solar Panel Defect Detection - 4 Classes

**Classes:** Bird-drop, Clean, Dusty, Physical-Damage

**Models:** YOLOv8m-cls & YOLOv11m-cls

**Goal:** Multi-class classification for physical defect detection

---

## 1. Install Ultralytics

In [1]:
import sys
print("Installing Ultralytics YOLO...")
!{sys.executable} -m pip install ultralytics -q
print("Installation complete!")

Installing Ultralytics YOLO...

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Installation complete!


## 2. Import Libraries

In [2]:
from ultralytics import YOLO
import os
import shutil
from pathlib import Path
import matplotlib.pyplot as plt
import cv2
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import json
from datetime import datetime

print("="*60)
print("Libraries Imported Successfully!")
print("="*60)
print(f"Working directory: {os.getcwd()}")

Libraries Imported Successfully!
Working directory: /home/compute.ashesi.lan/e.bilson/rgb_fault_detect


## 3. Dataset Configuration

In [3]:
# ========================================
# UPDATE THIS PATH TO YOUR DATASET!
# ========================================
SOURCE_DATASET = "rgb_fault_detect_data"  # CHANGE THIS!

# Target path for YOLO format
YOLO_DATASET = "./RGB_4Class_Detection"

# All 4 classes
CLASSES = ['Bird-drop', 'Clean', 'Dusty', 'Physical-Damage']
NUM_CLASSES = len(CLASSES)

print("="*60)
print("Dataset Configuration")
print("="*60)
print(f"Source: {SOURCE_DATASET}")
print(f"Target: {YOLO_DATASET}")
print(f"Classes: {NUM_CLASSES}")
for i, cls in enumerate(CLASSES):
    print(f"  {i}: {cls}")
print("="*60)

Dataset Configuration
Source: rgb_fault_detect_data
Target: ./RGB_4Class_Detection
Classes: 4
  0: Bird-drop
  1: Clean
  2: Dusty
  3: Physical-Damage


## 4. Prepare YOLO Classification Dataset

In [4]:
def prepare_yolo_cls_dataset(source_path, target_path, classes):
    """
    Organize dataset in YOLO-cls format
    
    Structure:
    target_path/
    ├── train/
    │   ├── Bird-drop/
    │   ├── Clean/
    │   ├── Dusty/
    │   └── Physical-Damage/
    ├── val/
    └── test/
    """
    print("\n" + "="*60)
    print("Preparing YOLO Classification Dataset")
    print("="*60)
    
    source_path = Path(source_path)
    target_path = Path(target_path)
    
    # Check if source exists
    if not source_path.exists():
        print(f"ERROR: Source path not found: {source_path}")
        print("Please update SOURCE_DATASET in Cell 3!")
        return None
    
    # Create target structure
    stats = {'train': {}, 'val': {}, 'test': {}}
    
    for split in ['train', 'val', 'test']:
        for cls in classes:
            target_dir = target_path / split / cls
            target_dir.mkdir(parents=True, exist_ok=True)
    
    # Copy images
    for split in ['train', 'val', 'test']:
        print(f"\nProcessing {split} split...")
        split_total = 0
        
        for cls in classes:
            source_dir = source_path / split / cls
            target_dir = target_path / split / cls
            
            if not source_dir.exists():
                print(f"  WARNING: {cls} not found in {split}!")
                stats[split][cls] = 0
                continue
            
            # Get all images
            images = []
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
                images.extend(source_dir.glob(ext))
                images.extend(source_dir.glob(ext.upper()))
            
            # Copy
            for img in images:
                shutil.copy2(img, target_dir / img.name)
            
            count = len(images)
            stats[split][cls] = count
            split_total += count
            print(f"  {cls:20s}: {count:4d} images")
        
        print(f"  {split.upper()} TOTAL: {split_total} images")
    
    return stats

# Prepare dataset
dataset_stats = prepare_yolo_cls_dataset(SOURCE_DATASET, YOLO_DATASET, CLASSES)

if dataset_stats:
    print("\n" + "="*60)
    print("Dataset Preparation Complete!")
    print("="*60)
    
    # Summary
    total_train = sum(dataset_stats['train'].values())
    total_val = sum(dataset_stats['val'].values())
    total_test = sum(dataset_stats['test'].values())
    total_all = total_train + total_val + total_test
    
    print(f"\nTotal Dataset: {total_all} images")
    print(f"  Train: {total_train}")
    print(f"  Val:   {total_val}")
    print(f"  Test:  {total_test}")
else:
    print("\nERROR: Dataset preparation failed!")
    print("Please check your SOURCE_DATASET path in Cell 3.")


Preparing YOLO Classification Dataset

Processing train split...
  Bird-drop           :  177 images
  Clean               :  169 images
  Dusty               :  162 images
  Physical-Damage     :  132 images
  TRAIN TOTAL: 640 images

Processing val split...
  Bird-drop           :  104 images
  Clean               :  102 images
  Dusty               :   97 images
  Physical-Damage     :   78 images
  VAL TOTAL: 381 images

Processing test split...
  Bird-drop           :   17 images
  Clean               :   18 images
  Dusty               :   16 images
  Physical-Damage     :   15 images
  TEST TOTAL: 66 images

Dataset Preparation Complete!

Total Dataset: 1087 images
  Train: 640
  Val:   381
  Test:  66


## 5. Train YOLOv8m-cls

In [5]:
print("="*60)
print("TRAINING YOLOv8m CLASSIFICATION")
print("="*60)
print(f"Classes: {NUM_CLASSES}")
print(f"Expected time: 5-10 minutes\n")

# Load YOLOv8m classification model
model_v8 = YOLO('yolov8m-cls.pt')

# Train
results_v8 = model_v8.train(
    data=YOLO_DATASET,
    epochs=100,
    imgsz=224,
    batch=32,
    patience=15,
    save=True,
    device=0,
    workers=4,
    project='runs_rgb',
    name='yolov8m_4class',
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    cos_lr=True,
    label_smoothing=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.5,
    mosaic=0.0,
    mixup=0.0
)

print("\n" + "="*60)
print("YOLOv8m Training Complete!")
print("="*60)

TRAINING YOLOv8m CLASSIFICATION
Classes: 4
Expected time: 5-10 minutes

New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.241 🚀 Python-3.12.11 torch-2.9.1+cu128 CUDA:0 (Quadro RTX 6000, 24023MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=./RGB_4Class_Detection, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_rat

## 6. Train YOLOv11m-cls

In [6]:
print("="*60)
print("TRAINING YOLOv11m CLASSIFICATION")
print("="*60)
print(f"Classes: {NUM_CLASSES}")
print(f"Expected time: 5-10 minutes\n")

# Load YOLOv11m classification model
model_v11 = YOLO('yolo11m-cls.pt')

# Train
results_v11 = model_v11.train(
    data=YOLO_DATASET,
    epochs=100,
    imgsz=224,
    batch=32,
    patience=15,
    save=True,
    device=0,
    workers=4,
    project='runs_rgb',
    name='yolov11m_4class',
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    cos_lr=True,
    label_smoothing=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.5,
    mosaic=0.0,
    mixup=0.0
)

print("\n" + "="*60)
print("YOLOv11m Training Complete!")
print("="*60)

TRAINING YOLOv11m CLASSIFICATION
Classes: 4
Expected time: 5-10 minutes

New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.241 🚀 Python-3.12.11 torch-2.9.1+cu128 CUDA:0 (Quadro RTX 6000, 24023MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=./RGB_4Class_Detection, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ra

## 7. Validate Both Models

In [7]:
print("="*60)
print("MODEL VALIDATION")
print("="*60)

# Load best models
best_v8 = YOLO('runs_rgb/yolov8m_4class/weights/best.pt')
best_v11 = YOLO('runs_rgb/yolov11m_4class/weights/best.pt')

# Validate YOLOv8
print("\nValidating YOLOv8m...")
metrics_v8 = best_v8.val(data=YOLO_DATASET)
acc_v8 = metrics_v8.top1
print(f"YOLOv8m Top-1 Accuracy: {acc_v8*100:.2f}%")

# Validate YOLOv11
print("\nValidating YOLOv11m...")
metrics_v11 = best_v11.val(data=YOLO_DATASET)
acc_v11 = metrics_v11.top1
print(f"YOLOv11m Top-1 Accuracy: {acc_v11*100:.2f}%")

# Comparison
print("\n" + "="*60)
print("COMPARISON")
print("="*60)
print(f"YOLOv8m:  {acc_v8*100:.2f}%")
print(f"YOLOv11m: {acc_v11*100:.2f}%")
diff = abs(acc_v11 - acc_v8) * 100
winner = 'YOLOv11m' if acc_v11 > acc_v8 else 'YOLOv8m'
print(f"\nWinner: {winner} (+{diff:.1f}%)")

MODEL VALIDATION

Validating YOLOv8m...
Ultralytics 8.3.241 🚀 Python-3.12.11 torch-2.9.1+cu128 CUDA:0 (Quadro RTX 6000, 24023MiB)
YOLOv8m-cls summary (fused): 42 layers, 15,767,780 parameters, 0 gradients, 41.6 GFLOPs
train: /home/compute.ashesi.lan/e.bilson/rgb_fault_detect/RGB_4Class_Detection/train... found 640 images in 4 classes ✅ 
val: /home/compute.ashesi.lan/e.bilson/rgb_fault_detect/RGB_4Class_Detection/val... found 381 images in 4 classes ✅ 
test: /home/compute.ashesi.lan/e.bilson/rgb_fault_detect/RGB_4Class_Detection/test... found 66 images in 4 classes ✅ 
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3226.4±2126.3 MB/s, size: 736.3 KB)
val: Scanning /home/compute.ashesi.lan/e.bilson/rgb_fault_detect/RGB_4Class_Detection/val... 381 images, 0 corrupt: 100% ━━━━━━━━━━━━ 381/381 978.6Kit/s 0.0s
val: /home/compute.ashesi.lan/e.bilson/rgb_fault_detect/RGB_4Class_Detection/val/Clean/Clean (13).jpg: corrupt JPEG restored and saved
val: /home/compute.ashesi.lan/e.bilson/rgb_faul

## 8. Test on Test Set

In [8]:
def test_model(model, test_folder, classes):
    """
    Test model on test set and collect predictions
    """
    all_preds = []
    all_labels = []
    all_confs = []
    
    test_folder = Path(test_folder)
    
    for class_idx, class_name in enumerate(classes):
        class_folder = test_folder / class_name
        
        if not class_folder.exists():
            continue
        
        # Get all images
        images = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
            images.extend(class_folder.glob(ext))
            images.extend(class_folder.glob(ext.upper()))
        
        for img_path in images:
            # Predict
            results = model(str(img_path), verbose=False)
            pred_class = results[0].probs.top1
            confidence = results[0].probs.top1conf.item()
            
            all_preds.append(pred_class)
            all_labels.append(class_idx)
            all_confs.append(confidence)
    
    return np.array(all_labels), np.array(all_preds), np.array(all_confs)

print("="*60)
print("TESTING ON TEST SET")
print("="*60)

test_folder = Path(YOLO_DATASET) / 'test'

# Test YOLOv8
print("\nTesting YOLOv8m...")
y_true_v8, y_pred_v8, conf_v8 = test_model(best_v8, test_folder, CLASSES)
acc_test_v8 = accuracy_score(y_true_v8, y_pred_v8)
print(f"Test Accuracy: {acc_test_v8*100:.2f}%")
print(f"Average Confidence: {conf_v8.mean()*100:.1f}%")

# Test YOLOv11
print("\nTesting YOLOv11m...")
y_true_v11, y_pred_v11, conf_v11 = test_model(best_v11, test_folder, CLASSES)
acc_test_v11 = accuracy_score(y_true_v11, y_pred_v11)
print(f"Test Accuracy: {acc_test_v11*100:.2f}%")
print(f"Average Confidence: {conf_v11.mean()*100:.1f}%")

print("\n" + "="*60)
print("TEST RESULTS")
print("="*60)
print(f"YOLOv8m:  {acc_test_v8*100:.2f}%")
print(f"YOLOv11m: {acc_test_v11*100:.2f}%")

TESTING ON TEST SET

Testing YOLOv8m...
Test Accuracy: 93.94%
Average Confidence: 97.1%

Testing YOLOv11m...
Test Accuracy: 98.48%
Average Confidence: 98.2%

TEST RESULTS
YOLOv8m:  93.94%
YOLOv11m: 98.48%


## 9. Classification Reports

In [9]:
print("="*60)
print("YOLOv8m Classification Report")
print("="*60)
print(classification_report(y_true_v8, y_pred_v8, target_names=CLASSES, digits=3))

print("\n" + "="*60)
print("YOLOv11m Classification Report")
print("="*60)
print(classification_report(y_true_v11, y_pred_v11, target_names=CLASSES, digits=3))

YOLOv8m Classification Report
                 precision    recall  f1-score   support

      Bird-drop      1.000     0.941     0.970        17
          Clean      0.895     0.944     0.919        18
          Dusty      0.875     0.875     0.875        16
Physical-Damage      1.000     1.000     1.000        15

       accuracy                          0.939        66
      macro avg      0.942     0.940     0.941        66
   weighted avg      0.941     0.939     0.940        66


YOLOv11m Classification Report
                 precision    recall  f1-score   support

      Bird-drop      1.000     1.000     1.000        17
          Clean      1.000     0.944     0.971        18
          Dusty      0.941     1.000     0.970        16
Physical-Damage      1.000     1.000     1.000        15

       accuracy                          0.985        66
      macro avg      0.985     0.986     0.985        66
   weighted avg      0.986     0.985     0.985        66



## 10. Confusion Matrices

In [10]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# YOLOv8 confusion matrix
cm_v8 = confusion_matrix(y_true_v8, y_pred_v8)
sns.heatmap(cm_v8, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax1,
            cbar_kws={'label': 'Count'})
ax1.set_title(f'YOLOv8m ({acc_test_v8*100:.1f}%)', fontsize=14, fontweight='bold', pad=15)
ax1.set_ylabel('True Label', fontsize=12)
ax1.set_xlabel('Predicted Label', fontsize=12)
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')

# YOLOv11 confusion matrix
cm_v11 = confusion_matrix(y_true_v11, y_pred_v11)
sns.heatmap(cm_v11, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax2,
            cbar_kws={'label': 'Count'})
ax2.set_title(f'YOLOv11m ({acc_test_v11*100:.1f}%)', fontsize=14, fontweight='bold', pad=15)
ax2.set_ylabel('True Label', fontsize=12)
ax2.set_xlabel('Predicted Label', fontsize=12)
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('confusion_matrices_4class.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved: confusion_matrices_4class.png")

<Figure size 1600x600 with 4 Axes>


Saved: confusion_matrices_4class.png


## 11. Per-Class Accuracy

In [11]:
# Calculate per-class accuracy for best model
best_model_name = 'YOLOv11m' if acc_test_v11 > acc_test_v8 else 'YOLOv8m'
y_true_best = y_true_v11 if acc_test_v11 > acc_test_v8 else y_true_v8
y_pred_best = y_pred_v11 if acc_test_v11 > acc_test_v8 else y_pred_v8
cm_best = cm_v11 if acc_test_v11 > acc_test_v8 else cm_v8

print("="*60)
print(f"Per-Class Accuracy ({best_model_name})")
print("="*60)

per_class_acc = []
for i, cls in enumerate(CLASSES):
    if cm_best[i].sum() > 0:
        acc = cm_best[i, i] / cm_best[i].sum() * 100
        per_class_acc.append(acc)
        print(f"{cls:20s}: {acc:5.1f}%  ({cm_best[i, i]}/{cm_best[i].sum()})")
    else:
        per_class_acc.append(0)
        print(f"{cls:20s}: N/A (no test samples)")

print(f"\nMean per-class accuracy: {np.mean(per_class_acc):.1f}%")

Per-Class Accuracy (YOLOv11m)
Bird-drop           : 100.0%  (17/17)
Clean               :  94.4%  (17/18)
Dusty               : 100.0%  (16/16)
Physical-Damage     : 100.0%  (15/15)

Mean per-class accuracy: 98.6%


## 12. Export for Raspberry Pi

In [12]:
print("="*60)
print("EXPORTING FOR RASPBERRY PI")
print("="*60)

# Choose best model
if acc_test_v11 > acc_test_v8:
    best_model = best_v11
    model_name = 'yolov11m'
    best_acc = acc_test_v11
else:
    best_model = best_v8
    model_name = 'yolov8m'
    best_acc = acc_test_v8

print(f"\nBest model: {model_name}")
print(f"Test accuracy: {best_acc*100:.2f}%")

# Export to ONNX (CPU optimized)
print("\nExporting to ONNX format...")
try:
    best_model.export(format='onnx', imgsz=224, simplify=True)
    print("ONNX export successful!")
except Exception as e:
    print(f"ONNX export failed: {e}")

# Copy best.pt for easy access
import shutil
best_pt_source = f'runs_rgb/{model_name}_4class/weights/best.pt'
shutil.copy2(best_pt_source, 'best_rgb_4class.pt')
print(f"\nCopied best model to: best_rgb_4class.pt")

print("\n" + "="*60)
print("Export Complete!")
print("="*60)

EXPORTING FOR RASPBERRY PI

Best model: yolov11m
Test accuracy: 98.48%

Exporting to ONNX format...
Ultralytics 8.3.241 🚀 Python-3.12.11 torch-2.9.1+cu128 CPU (Intel Xeon Gold 5217 CPU @ 3.00GHz)

PyTorch: starting from 'runs_rgb/yolov11m_4class/weights/best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 4) (19.9 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
WARNING ⚠️ Retry 1/2 failed: Command 'pip install --no-cache-dir "onnx>=1.12.0,<2.0.0" "onnxslim>=0.1.71" "onnxruntime-gpu" ' returned non-zero exit status 1.
WARNING ⚠️ Retry 2/2 failed: Command 'pip install --no-cache-dir "onnx>=1.12.0,<2.0.0" "onnxslim>=0.1.71" "onnxruntime-gpu" ' returned non-zero exit status 1.
WARNING ⚠️ requirements: ❌ Command 'pip install --no-cache-dir "onnx>=1.12.0,<2.0.0" "onnxslim>=0.1.71" "onnxruntime-gpu" ' returned non-zero exit status 1.
ERROR ❌ ONNX: export failure 35.4s: No module na

## 13. Create Raspberry Pi Deployment Script

In [13]:
deployment_code = f'''#!/usr/bin/env python3
"""
RGB Solar Panel Defect Detector - Raspberry Pi
Model: {model_name} ({best_acc*100:.1f}% accuracy)
Classes: Bird-drop, Clean, Dusty, Physical-Damage
"""

from ultralytics import YOLO
import cv2
import time
import numpy as np

# Configuration
MODEL_PATH = 'best_rgb_4class.pt'  # or best.onnx for faster inference
CLASSES = {CLASSES}

# Load model
print("Loading model...")
model = YOLO(MODEL_PATH)
print(f"Model loaded: {{MODEL_PATH}}")
print(f"Classes: {{len(CLASSES)}}\\n")

def detect_defect(image_path):
    """
    Detect defect in solar panel image
    
    Returns:
        class_name: Detected class
        confidence: Prediction confidence (0-1)
        all_probs: Probabilities for all classes
    """
    results = model(image_path, verbose=False)
    probs = results[0].probs
    
    class_id = probs.top1
    confidence = probs.top1conf.item()
    class_name = CLASSES[class_id]
    all_probs = probs.data.cpu().numpy()
    
    return class_name, confidence, all_probs

def print_detection(class_name, confidence, all_probs):
    """
    Print detection results
    """
    print(f"\\nDetection: {{class_name}}")
    print(f"Confidence: {{confidence*100:.1f}}%")
    print("\\nAll probabilities:")
    for i, cls in enumerate(CLASSES):
        prob = all_probs[i] * 100
        bar = "=" * int(prob / 2)
        print(f"  {{cls:20s}}: {{prob:5.1f}}% {{bar}}")

def camera_stream(camera_id=0):
    """
    Real-time defect detection from camera
    """
    cap = cv2.VideoCapture(camera_id)
    print(f"\\nCamera started. Press 'q' to quit.")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Save temp frame
        cv2.imwrite('temp_frame.jpg', frame)
        
        # Detect
        start_time = time.time()
        class_name, confidence, _ = detect_defect('temp_frame.jpg')
        inference_time = (time.time() - start_time) * 1000
        
        # Color based on class
        if class_name == 'Clean':
            color = (0, 255, 0)  # Green
        elif class_name == 'Physical-Damage':
            color = (0, 0, 255)  # Red
        else:
            color = (0, 165, 255)  # Orange
        
        # Draw results
        text = f"{{class_name}}: {{confidence*100:.1f}}%"
        cv2.putText(frame, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                   0.8, color, 2)
        cv2.putText(frame, f"{{inference_time:.0f}}ms", (10, 60),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        cv2.imshow('RGB Defect Detector', frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()

def batch_process(folder_path):
    """
    Process all images in a folder
    """
    import os
    from pathlib import Path
    
    folder = Path(folder_path)
    images = []
    for ext in ['*.jpg', '*.jpeg', '*.png']:
        images.extend(folder.glob(ext))
    
    print(f"\\nProcessing {{len(images)}} images...\\n")
    
    results_summary = {{cls: 0 for cls in CLASSES}}
    
    for img_path in images:
        class_name, confidence, _ = detect_defect(str(img_path))
        results_summary[class_name] += 1
        print(f"{{img_path.name:30s}} -> {{class_name:20s}} ({{confidence*100:.1f}}%)")
    
    print("\\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    for cls, count in results_summary.items():
        print(f"{{cls:20s}}: {{count}}")

if __name__ == "__main__":
    import sys
    
    print("="*60)
    print("RGB Solar Panel Defect Detector")
    print("="*60)
    print(f"Model: {{MODEL_PATH}}")
    print(f"Accuracy: {best_acc*100:.1f}%")
    print("="*60)
    
    if len(sys.argv) > 1:
        path = sys.argv[1]
        if os.path.isfile(path):
            # Single image
            class_name, confidence, all_probs = detect_defect(path)
            print_detection(class_name, confidence, all_probs)
        elif os.path.isdir(path):
            # Folder
            batch_process(path)
    else:
        # Demo
        print("\\nUsage:")
        print("  Single image: python3 raspberry_pi_rgb.py image.jpg")
        print("  Batch:        python3 raspberry_pi_rgb.py folder/")
        print("  Camera:       Uncomment camera_stream() below")
        print("\\n")
        
        # Uncomment for live camera
        # camera_stream()
'''

with open('raspberry_pi_rgb.py', 'w') as f:
    f.write(deployment_code)

print("Created: raspberry_pi_rgb.py")
print("\nDeployment files ready for Raspberry Pi!")

Created: raspberry_pi_rgb.py

Deployment files ready for Raspberry Pi!


## 14. Final Summary

In [14]:
# Save results
results_dict = {
    'timestamp': datetime.now().isoformat(),
    'classes': CLASSES,
    'num_classes': NUM_CLASSES,
    'models': {
        'yolov8m': {
            'val_accuracy': float(acc_v8),
            'test_accuracy': float(acc_test_v8)
        },
        'yolov11m': {
            'val_accuracy': float(acc_v11),
            'test_accuracy': float(acc_test_v11)
        }
    },
    'best_model': model_name,
    'best_accuracy': float(best_acc),
    'dataset_stats': dataset_stats
}

with open('rgb_training_results.json', 'w') as f:
    json.dump(results_dict, f, indent=2)

print("="*60)
print("TRAINING COMPLETE!")
print("="*60)

print(f"\nClasses: {NUM_CLASSES}")
for i, cls in enumerate(CLASSES):
    print(f"  {i}: {cls}")

print(f"\nResults:")
print(f"  YOLOv8m:  {acc_test_v8*100:.2f}%")
print(f"  YOLOv11m: {acc_test_v11*100:.2f}%")
print(f"\n  Best: {model_name} ({best_acc*100:.2f}%)")

print(f"\nFiles Created:")
print("  Models:")
print("    - runs_rgb/yolov8m_4class/weights/best.pt")
print("    - runs_rgb/yolov11m_4class/weights/best.pt")
print("    - best_rgb_4class.pt")
print("    - best.onnx")
print("  Deployment:")
print("    - raspberry_pi_rgb.py")
print("  Results:")
print("    - confusion_matrices_4class.png")
print("    - rgb_training_results.json")

print(f"\nNext Steps:")
print("  1. Test on Raspberry Pi")
print("  2. Verify real-time performance")
print("  3. Integrate with thermal model (86%)")
print("  4. Build dual-stream fusion system")

print("\n" + "="*60)
print(f"RGB Stream Ready! ({best_acc*100:.1f}% accuracy)")
print("="*60)

TRAINING COMPLETE!

Classes: 4
  0: Bird-drop
  1: Clean
  2: Dusty
  3: Physical-Damage

Results:
  YOLOv8m:  93.94%
  YOLOv11m: 98.48%

  Best: yolov11m (98.48%)

Files Created:
  Models:
    - runs_rgb/yolov8m_4class/weights/best.pt
    - runs_rgb/yolov11m_4class/weights/best.pt
    - best_rgb_4class.pt
    - best.onnx
  Deployment:
    - raspberry_pi_rgb.py
  Results:
    - confusion_matrices_4class.png
    - rgb_training_results.json

Next Steps:
  1. Test on Raspberry Pi
  2. Verify real-time performance
  3. Integrate with thermal model (86%)
  4. Build dual-stream fusion system

RGB Stream Ready! (98.5% accuracy)


In [15]:
# ============================================================
#  MODEL ASSESSMENT — RGB Solar Panel Defect Detection
#  YOLOv8m-cls  vs  YOLOv11m-cls
# ============================================================

# ── UPDATE THESE PATHS ──────────────────────────────────────
V8_WEIGHTS  = "runs_rgb/yolov8m_4class/weights/best.pt"
V11_WEIGHTS = "runs_rgb/yolov11m_4class/weights/best.pt"
TEST_DIR    = "RGB_4Class_Detection/test"
V8_RUN_DIR  = "runs_rgb/yolov8m_4class"
V11_RUN_DIR = "runs_rgb/yolov11m_4class"
CLASSES     = ["Bird-drop", "Clean", "Dusty", "Physical-Damage"]
# ────────────────────────────────────────────────────────────

import os, warnings
warnings.filterwarnings("ignore")
os.makedirs("assessment_outputs", exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

# ── Style ────────────────────────────────────────────────────
BG       = "#FFFFFF"
PANEL    = "#F7F9FC"
BORDER   = "#D0D7DE"
TEXT_PRI = "#1A1A2E"
TEXT_SEC = "#555F6E"
C_V8     = "#1565C0"
C_V11    = "#C62828"
C_GOOD   = "#2E7D32"
C_BAD    = "#B71C1C"
C_ACCENT = "#E65100"

plt.rcParams.update({
    "figure.facecolor": BG,    "axes.facecolor"  : PANEL,
    "axes.edgecolor"  : BORDER,"axes.labelcolor" : TEXT_PRI,
    "axes.titlecolor" : TEXT_PRI,"axes.grid"     : True,
    "axes.grid.which" : "major","grid.color"     : BORDER,
    "grid.linewidth"  : 0.6,   "grid.alpha"      : 0.6,
    "xtick.color"     : TEXT_SEC,"ytick.color"   : TEXT_SEC,
    "xtick.labelsize" : 9,     "ytick.labelsize" : 9,
    "legend.facecolor": "#FFFFFF","legend.edgecolor": BORDER,
    "legend.labelcolor": TEXT_PRI,"legend.fontsize": 9,
    "text.color"      : TEXT_PRI,"font.family"    : ["DejaVu Sans"],
    "figure.dpi"      : 130,   "savefig.dpi"     : 180,
    "savefig.facecolor": BG,   "savefig.bbox"    : "tight",
    "lines.linewidth" : 2.0,
})

cmap_v8   = LinearSegmentedColormap.from_list("cb", ["#EBF5FB", C_V8],  N=256)
cmap_v11  = LinearSegmentedColormap.from_list("co", ["#FDEDEC", C_V11], N=256)
cmap_heat = LinearSegmentedColormap.from_list("ht", ["#FDFEFE","#AED6F1","#2980B9","#1A5276"], N=256)

def add_watermark(fig):
    fig.text(0.99, 0.005, f"Solar Panel Defect Assessment  •  {datetime.now():%Y-%m-%d}",
             ha="right", va="bottom", color=TEXT_SEC, fontsize=7, alpha=0.7)

def save_fig(fig, name):
    fig.savefig(f"assessment_outputs/{name}.png", facecolor=BG)
    plt.show()
    print(f"  ✓  Saved → assessment_outputs/{name}.png")

# ── Inference ────────────────────────────────────────────────
def run_inference(weights, test_dir, classes):
    model = YOLO(weights)
    y_true, y_pred, y_conf = [], [], []
    for idx, cls in enumerate(classes):
        folder = Path(test_dir) / cls
        if not folder.exists():
            print(f"  WARNING: {folder} not found"); continue
        images = []
        for ext in ("*.jpg","*.jpeg","*.png","*.bmp","*.JPG","*.JPEG","*.PNG"):
            images.extend(folder.glob(ext))
        for img in images:
            res = model(str(img), verbose=False)
            y_true.append(idx)
            y_pred.append(int(res[0].probs.top1))
            y_conf.append(float(res[0].probs.top1conf))
    return np.array(y_true), np.array(y_pred), np.array(y_conf)

print("Running YOLOv8m  inference …")
y_true_v8,  y_pred_v8,  conf_v8  = run_inference(V8_WEIGHTS,  TEST_DIR, CLASSES)
acc_v8  = accuracy_score(y_true_v8,  y_pred_v8)
print(f"  YOLOv8m  → {acc_v8*100:.2f}%  ({(y_true_v8==y_pred_v8).sum()}/{len(y_true_v8)} correct)")

print("Running YOLOv11m inference …")
y_true_v11, y_pred_v11, conf_v11 = run_inference(V11_WEIGHTS, TEST_DIR, CLASSES)
acc_v11 = accuracy_score(y_true_v11, y_pred_v11)
print(f"  YOLOv11m → {acc_v11*100:.2f}%  ({(y_true_v11==y_pred_v11).sum()}/{len(y_true_v11)} correct)")

# ── Classification Reports ───────────────────────────────────
for y_true, y_pred, lbl in [(y_true_v8, y_pred_v8,"YOLOv8m"),(y_true_v11,y_pred_v11,"YOLOv11m")]:
    print("="*60)
    print(f"  {lbl} — Classification Report")
    print("="*60)
    print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

# ── Pre-compute shared metrics ───────────────────────────────
def get_per_class(y_true, y_pred):
    p  = precision_score(y_true, y_pred, average=None, zero_division=0)
    r  = recall_score   (y_true, y_pred, average=None, zero_division=0)
    f1 = f1_score       (y_true, y_pred, average=None, zero_division=0)
    return np.stack([p, r, f1])

m_v8  = get_per_class(y_true_v8,  y_pred_v8)
m_v11 = get_per_class(y_true_v11, y_pred_v11)

kpis = {}
for y_true, y_pred, conf, lbl in [
    (y_true_v8,  y_pred_v8,  conf_v8,  "YOLOv8m"),
    (y_true_v11, y_pred_v11, conf_v11, "YOLOv11m"),
]:
    kpis[lbl] = {
        "Accuracy" : accuracy_score (y_true, y_pred)                                     * 100,
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0)   * 100,
        "Recall"   : recall_score   (y_true, y_pred, average="macro", zero_division=0)   * 100,
        "F1"       : f1_score       (y_true, y_pred, average="macro", zero_division=0)   * 100,
        "Avg Conf" : conf.mean()                                                          * 100,
    }

# ════════════════════════════════════════════════════════════
#  FIGURE 1 — Training Curves
# ════════════════════════════════════════════════════════════
def load_csv(run_dir):
    p = Path(run_dir) / "results.csv"
    if not p.exists(): print(f"  WARNING: {p} not found"); return None
    df = pd.read_csv(p); df.columns = [c.strip() for c in df.columns]
    return df

def pick(df, keys):
    for k in keys:
        if k in df.columns: return df[k].values
    return None

col_map = {
    "train_loss": ["train/loss",  "train/cls_loss"],
    "val_loss"  : ["val/loss",    "val/cls_loss"],
    "top1"      : ["metrics/accuracy_top1", "top1"],
    "top5"      : ["metrics/accuracy_top5", "top5"],
    "lr"        : ["lr/pg0", "lr0"],
}

df_v8  = load_csv(V8_RUN_DIR)
df_v11 = load_csv(V11_RUN_DIR)

fig = plt.figure(figsize=(18, 10))
fig.suptitle("Training Curves — YOLOv8m vs YOLOv11m",
             fontsize=16, fontweight="bold", color=TEXT_PRI, y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

panel_specs = [
    (0,0,"Train Loss",         "train_loss","Loss",    None,     False),
    (0,1,"Validation Loss",    "val_loss",  "Loss",    None,     False),
    (0,2,"Learning Rate",      "lr",        "LR",      None,     False),
    (1,0,"Val Top-1 Accuracy", "top1",      "Accuracy",(0,1.05), True),
    (1,1,"Val Top-5 Accuracy", "top5",      "Accuracy",(0,1.05), True),
]
for r,c,title,key,ylabel,ylim,pct in panel_specs:
    ax = fig.add_subplot(gs[r,c]); ax.set_facecolor(PANEL)
    for df, color, lbl in [(df_v8,C_V8,"YOLOv8m"),(df_v11,C_V11,"YOLOv11m")]:
        if df is None: continue
        y = pick(df, col_map[key])
        if y is None: continue
        x = np.arange(1, len(y)+1)
        ax.plot(x, y, color=color, linewidth=2.2, label=lbl, zorder=3)
        if len(y) > 6:
            sm = pd.Series(y).rolling(5, min_periods=1).mean().values
            ax.plot(x, sm, color=color, linewidth=1.0, linestyle="--", alpha=0.45, zorder=2)
    ax.set_title(title, fontsize=11, fontweight="bold", pad=7)
    ax.set_xlabel("Epoch", color=TEXT_SEC, fontsize=9)
    ax.set_ylabel(ylabel + (" (%)" if pct else ""), color=TEXT_SEC, fontsize=9)
    if ylim: ax.set_ylim(ylim)
    if pct: ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"{v*100:.0f}%"))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))
    ax.legend(); ax.spines[["top","right"]].set_visible(False)

ax_bar = fig.add_subplot(gs[1,2]); ax_bar.set_facecolor(PANEL)
labels, vals, colors = [], [], []
for df, color, lbl in [(df_v8,C_V8,"YOLOv8m"),(df_v11,C_V11,"YOLOv11m")]:
    if df is None: continue
    y = pick(df, col_map["top1"])
    if y is None: continue
    labels.append(lbl); vals.append(float(np.max(y))*100); colors.append(color)
if vals:
    bars = ax_bar.bar(labels, vals, color=colors, width=0.45, edgecolor=BORDER, zorder=3)
    for bar, val in zip(bars, vals):
        ax_bar.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                    f"{val:.2f}%", ha="center", va="bottom",
                    color=C_ACCENT, fontsize=11, fontweight="bold")
    ax_bar.set_ylim(0, 110)
    ax_bar.set_title("Best Val Top-1 Accuracy", fontsize=11, fontweight="bold", pad=7)
    ax_bar.set_ylabel("Accuracy (%)", color=TEXT_SEC, fontsize=9)
    ax_bar.spines[["top","right"]].set_visible(False)
    ax_bar.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"{v:.0f}%"))

plt.tight_layout(); add_watermark(fig); save_fig(fig, "01_training_curves")

# ════════════════════════════════════════════════════════════
#  FIGURE 2 — Confusion Matrices
# ════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Confusion Matrices — Test Set",
             fontsize=15, fontweight="bold", color=TEXT_PRI, y=1.01)

for ax, y_true, y_pred, acc, lbl, cmap, accent in [
    (axes[0], y_true_v8,  y_pred_v8,  acc_v8,  "YOLOv8m",  cmap_v8,  C_V8),
    (axes[1], y_true_v11, y_pred_v11, acc_v11, "YOLOv11m", cmap_v11, C_V11),
]:
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, ax=ax, annot=False, cmap=cmap, vmin=0, vmax=1,
                linewidths=0.5, linecolor=BORDER,
                cbar_kws={"shrink":0.78,"label":"Recall proportion"})
    ax.collections[0].colorbar.ax.yaxis.label.set_color(TEXT_SEC)
    ax.collections[0].colorbar.ax.tick_params(colors=TEXT_SEC)
    n = len(CLASSES)
    for i in range(n):
        for j in range(n):
            raw = cm[i,j]; norm = cm_norm[i,j]
            ax.text(j+0.5, i+0.5, f"{norm:.2f}\n({raw})",
                    ha="center", va="center", fontsize=9, fontweight="bold",
                    color="white" if norm > 0.55 else TEXT_PRI)
    tick_lbl = [c.replace("-","-\n") for c in CLASSES]
    ax.set_xticks(np.arange(n)+0.5); ax.set_yticks(np.arange(n)+0.5)
    ax.set_xticklabels(tick_lbl, rotation=0, ha="center", color=TEXT_PRI, fontsize=9)
    ax.set_yticklabels(tick_lbl, rotation=0, va="center", color=TEXT_PRI, fontsize=9)
    ax.set_xlabel("Predicted Class", fontsize=11, color=TEXT_SEC, labelpad=8)
    ax.set_ylabel("True Class",      fontsize=11, color=TEXT_SEC, labelpad=8)
    ax.set_title(f"{lbl}  —  Test Accuracy: {acc*100:.2f}%",
                 fontsize=13, fontweight="bold", color=accent, pad=12)
    ax.tick_params(length=0)

plt.tight_layout(pad=2.5); add_watermark(fig); save_fig(fig, "02_confusion_matrices")

# ════════════════════════════════════════════════════════════
#  FIGURE 3 — Per-Class Precision / Recall / F1
# ════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Per-Class Metrics — YOLOv8m vs YOLOv11m (Test Set)",
             fontsize=15, fontweight="bold", color=TEXT_PRI, y=1.02)

x = np.arange(len(CLASSES)); w = 0.35
for ax, mi, name in zip(axes, range(3), ["Precision","Recall","F1-Score"]):
    b8  = ax.bar(x-w/2, m_v8[mi]*100,  width=w, color=C_V8,  label="YOLOv8m",
                 edgecolor=BORDER, linewidth=0.6, zorder=3, alpha=0.90)
    b11 = ax.bar(x+w/2, m_v11[mi]*100, width=w, color=C_V11, label="YOLOv11m",
                 edgecolor=BORDER, linewidth=0.6, zorder=3, alpha=0.90)
    for bars in (b8, b11):
        for bar in bars:
            h = bar.get_height()
            if h > 3:
                ax.text(bar.get_x()+bar.get_width()/2, h+0.8, f"{h:.1f}",
                        ha="center", va="bottom", color=TEXT_PRI, fontsize=7.5, fontweight="bold")
    ax.set_title(name, fontsize=13, fontweight="bold", pad=8)
    ax.set_xticks(x)
    ax.set_xticklabels(CLASSES, rotation=20, ha="right", color=TEXT_PRI, fontsize=9)
    ax.set_ylabel(f"{name} (%)", color=TEXT_SEC, fontsize=10)
    ax.set_ylim(0, 112)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"{v:.0f}%"))
    ax.legend(loc="lower right"); ax.spines[["top","right"]].set_visible(False)
    ax.axhline(100, color=BORDER, linewidth=0.8, linestyle="--", alpha=0.7)

plt.tight_layout(pad=2.0); add_watermark(fig); save_fig(fig, "03_per_class_metrics")

# ════════════════════════════════════════════════════════════
#  FIGURE 4 — Confidence Distributions
# ════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Prediction Confidence Distributions (Test Set)",
             fontsize=15, fontweight="bold", color=TEXT_PRI, y=1.01)
axes = axes.flatten(); bins = np.linspace(0, 1, 25)

for i, cls_name in enumerate(CLASSES):
    ax = axes[i]
    for (y_true, y_pred, conf), color, lbl in [
        ((y_true_v8,  y_pred_v8,  conf_v8),  C_V8,  "YOLOv8m"),
        ((y_true_v11, y_pred_v11, conf_v11), C_V11, "YOLOv11m"),
    ]:
        mask = y_true == i
        if mask.sum() == 0: continue
        correct = conf[mask & (y_pred == i)]
        wrong   = conf[mask & (y_pred != i)]
        if len(correct):
            ax.hist(correct, bins=bins, color=color, alpha=0.60,
                    label=f"{lbl} ✓", edgecolor="white", linewidth=0.3, zorder=3)
        if len(wrong):
            ax.hist(wrong, bins=bins, color=C_BAD, alpha=0.45,
                    label=f"{lbl} ✗", edgecolor="white", linewidth=0.3,
                    histtype="stepfilled", hatch="//", zorder=2)
    ax.set_title(cls_name, fontsize=12, fontweight="bold", pad=7)
    ax.set_xlabel("Confidence Score", color=TEXT_SEC, fontsize=9)
    ax.set_ylabel("Count", color=TEXT_SEC, fontsize=9)
    ax.set_xlim(0, 1)
    ax.legend(fontsize=7.5); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout(pad=2.5); add_watermark(fig); save_fig(fig, "04_confidence_distributions")

# ════════════════════════════════════════════════════════════
#  FIGURE 5 — Summary Dashboard
# ════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(20, 13))
fig.patch.set_facecolor(BG)
fig.suptitle("Model Assessment Summary — RGB Solar Panel Defect Detection",
             fontsize=18, fontweight="bold", color=TEXT_PRI, y=0.99)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.55, wspace=0.4)

# KPI cards
kpi_keys = ["Accuracy","Precision","Recall","F1"]
for col, key in enumerate(kpi_keys):
    ax = fig.add_subplot(gs[0, col])
    ax.set_facecolor("#EAF2FB"); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
    v8_val  = kpis["YOLOv8m"][key]
    v11_val = kpis["YOLOv11m"][key]
    winner  = C_V11 if v11_val >= v8_val else C_V8
    ax.text(0.5, 0.88, key,       ha="center", va="top", color=TEXT_SEC,
            fontsize=10, fontweight="bold", transform=ax.transAxes)
    ax.text(0.5, 0.60, f"{v8_val:.1f}%",  ha="center", va="top", color=C_V8,
            fontsize=19, fontweight="bold", transform=ax.transAxes)
    ax.text(0.5, 0.35, f"{v11_val:.1f}%", ha="center", va="top", color=C_V11,
            fontsize=19, fontweight="bold", transform=ax.transAxes)
    ax.add_patch(mpatches.FancyBboxPatch(
        (0.02,0.02),0.96,0.96, boxstyle="round,pad=0.02",
        linewidth=2, edgecolor=winner, facecolor="none", transform=ax.transAxes))

for x_pos, lbl, col in [(0.25,"■ YOLOv8m",C_V8),(0.75,"■ YOLOv11m",C_V11)]:
    fig.text(x_pos, 0.765, lbl, ha="center", color=col, fontsize=10, fontweight="bold")

# Radar
ax_radar = fig.add_subplot(gs[1,:2], polar=True); ax_radar.set_facecolor(PANEL)
r_labels = ["Accuracy","Precision","Recall","F1","Avg Conf"]
angles = np.linspace(0, 2*np.pi, len(r_labels), endpoint=False).tolist() + [0]
for model_lbl, color in [("YOLOv8m",C_V8),("YOLOv11m",C_V11)]:
    vals = [kpis[model_lbl][k]/100 for k in r_labels] + [kpis[model_lbl][r_labels[0]]/100]
    ax_radar.plot(angles, vals, color=color, linewidth=2.2, label=model_lbl)
    ax_radar.fill(angles, vals, color=color, alpha=0.12)
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(r_labels, color=TEXT_PRI, fontsize=9)
ax_radar.set_yticklabels(["20","40","60","80","100"], color=TEXT_SEC, fontsize=7)
ax_radar.set_ylim(0,1)
ax_radar.set_title("Overall Metrics Radar", color=TEXT_PRI, fontsize=12, fontweight="bold", pad=15)
ax_radar.legend(loc="upper right", bbox_to_anchor=(1.28,1.12))
ax_radar.grid(color=BORDER, alpha=0.6); ax_radar.spines["polar"].set_edgecolor(BORDER)

# F1 Heatmap
ax_heat = fig.add_subplot(gs[1,2:]); ax_heat.set_facecolor(PANEL)
f1_v8  = f1_score(y_true_v8,  y_pred_v8,  average=None, zero_division=0)
f1_v11 = f1_score(y_true_v11, y_pred_v11, average=None, zero_division=0)
heat_df = pd.DataFrame(np.vstack([f1_v8,f1_v11])*100,
                       index=["YOLOv8m","YOLOv11m"], columns=CLASSES)
sns.heatmap(heat_df, ax=ax_heat, annot=True, fmt=".1f", cmap=cmap_heat,
            vmin=0, vmax=100, linewidths=0.8, linecolor=BORDER,
            annot_kws={"size":11,"weight":"bold"},
            cbar_kws={"shrink":0.85,"label":"F1-Score (%)"})
ax_heat.collections[0].colorbar.ax.yaxis.label.set_color(TEXT_SEC)
ax_heat.collections[0].colorbar.ax.tick_params(colors=TEXT_SEC)
ax_heat.set_title("Per-Class F1-Score Heatmap (%)", color=TEXT_PRI,
                  fontsize=12, fontweight="bold", pad=10)
ax_heat.set_xticklabels(CLASSES, color=TEXT_PRI, rotation=20, ha="right", fontsize=9)
ax_heat.set_yticklabels(["YOLOv8m","YOLOv11m"], color=TEXT_PRI, rotation=0, fontsize=10)
ax_heat.tick_params(length=0)

# Overall grouped bar
ax_grp = fig.add_subplot(gs[2,:3]); ax_grp.set_facecolor(PANEL)
metric_names = ["Accuracy","Precision","Recall","F1-Score"]
v8_vals  = [kpis["YOLOv8m"]["Accuracy"],  kpis["YOLOv8m"]["Precision"],
            kpis["YOLOv8m"]["Recall"],    kpis["YOLOv8m"]["F1"]]
v11_vals = [kpis["YOLOv11m"]["Accuracy"], kpis["YOLOv11m"]["Precision"],
            kpis["YOLOv11m"]["Recall"],   kpis["YOLOv11m"]["F1"]]
xp = np.arange(len(metric_names)); w = 0.36
b8  = ax_grp.bar(xp-w/2, v8_vals,  width=w, color=C_V8,  label="YOLOv8m",
                 edgecolor=BORDER, linewidth=0.6, zorder=3)
b11 = ax_grp.bar(xp+w/2, v11_vals, width=w, color=C_V11, label="YOLOv11m",
                 edgecolor=BORDER, linewidth=0.6, zorder=3)
for bars in (b8, b11):
    for bar in bars:
        h = bar.get_height()
        ax_grp.text(bar.get_x()+bar.get_width()/2, h+0.4, f"{h:.1f}%",
                    ha="center", va="bottom", color=C_ACCENT, fontsize=9, fontweight="bold")
ax_grp.set_xticks(xp); ax_grp.set_xticklabels(metric_names, color=TEXT_PRI, fontsize=11)
ax_grp.set_ylabel("Score (%)", color=TEXT_SEC, fontsize=10); ax_grp.set_ylim(0, 115)
ax_grp.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"{v:.0f}%"))
ax_grp.set_title("Overall Model Performance Comparison",
                 fontsize=12, fontweight="bold", color=TEXT_PRI, pad=8)
ax_grp.legend(); ax_grp.spines[["top","right"]].set_visible(False)

# Confidence violin
ax_vio = fig.add_subplot(gs[2,3]); ax_vio.set_facecolor(PANEL)
c_v8  = conf_v8 [y_true_v8  == y_pred_v8]  * 100
c_v11 = conf_v11[y_true_v11 == y_pred_v11] * 100
vparts = ax_vio.violinplot([c_v8, c_v11], positions=[0,1], widths=0.55,
                            showmeans=True, showmedians=True, showextrema=True)
for i, pc in enumerate(vparts["bodies"]):
    pc.set_facecolor([C_V8,C_V11][i]); pc.set_alpha(0.45)
for part in ("cmeans","cmedians","cbars","cmins","cmaxes"):
    if part in vparts:
        vparts[part].set_color(BORDER); vparts[part].set_linewidth(1.5)
ax_vio.set_xticks([0,1])
ax_vio.set_xticklabels(["YOLOv8m","YOLOv11m"], color=TEXT_PRI, fontsize=10)
ax_vio.set_ylabel("Confidence (%) — Correct Preds", color=TEXT_SEC, fontsize=8)
ax_vio.set_title("Confidence Distribution\n(Correct Predictions)",
                 fontsize=10, fontweight="bold", color=TEXT_PRI, pad=7)
ax_vio.set_ylim(0, 105); ax_vio.spines[["top","right"]].set_visible(False)
ax_vio.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"{v:.0f}%"))

add_watermark(fig); save_fig(fig, "00_summary_dashboard")

# ── Final summary print ──────────────────────────────────────
winner = "YOLOv11m" if acc_v11 > acc_v8 else "YOLOv8m"
print("\n" + "="*55)
print("  ASSESSMENT COMPLETE")
print("="*55)
print(f"  YOLOv8m  test accuracy : {acc_v8 *100:.2f}%")
print(f"  YOLOv11m test accuracy : {acc_v11*100:.2f}%")
print(f"  Winner                 : {winner}  (+{abs(acc_v11-acc_v8)*100:.2f}%)")
print(f"  All figures saved to   : ./assessment_outputs/")
print("="*55)

Running YOLOv8m  inference …
  YOLOv8m  → 93.94%  (62/66 correct)
Running YOLOv11m inference …
  YOLOv11m → 98.48%  (65/66 correct)
  YOLOv8m — Classification Report
                 precision    recall  f1-score   support

      Bird-drop      1.000     0.941     0.970        17
          Clean      0.895     0.944     0.919        18
          Dusty      0.875     0.875     0.875        16
Physical-Damage      1.000     1.000     1.000        15

       accuracy                          0.939        66
      macro avg      0.942     0.940     0.941        66
   weighted avg      0.941     0.939     0.940        66

  YOLOv11m — Classification Report
                 precision    recall  f1-score   support

      Bird-drop      1.000     1.000     1.000        17
          Clean      1.000     0.944     0.971        18
          Dusty      0.941     1.000     0.970        16
Physical-Damage      1.000     1.000     1.000        15

       accuracy                          0.985       

<Figure size 2340x1300 with 6 Axes>

  ✓  Saved → assessment_outputs/01_training_curves.png


<Figure size 2080x910 with 4 Axes>

  ✓  Saved → assessment_outputs/02_confusion_matrices.png


...

  ✓  Saved → assessment_outputs/03_per_class_metrics.png


<Figure size 1820x1300 with 4 Axes>

  ✓  Saved → assessment_outputs/04_confidence_distributions.png


<Figure size 2600x1690 with 9 Axes>

  ✓  Saved → assessment_outputs/00_summary_dashboard.png

  ASSESSMENT COMPLETE
  YOLOv8m  test accuracy : 93.94%
  YOLOv11m test accuracy : 98.48%
  Winner                 : YOLOv11m  (+4.55%)
  All figures saved to   : ./assessment_outputs/
